<a href="https://colab.research.google.com/github/KalinaMarkova/deep_learning_course_project/blob/main/02_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import os
import glob
import shutil
from tqdm import tqdm
import json
from google.colab import drive
from datasets import Dataset
from transformers import AutoTokenizer

In [5]:
!pip install -q datasets

In [8]:
!pip install -q --upgrade transformers tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 33.3 MB/s eta 0:00:00


# Fine-Grained Analysis of Propaganda in News Articles
## Notebook 02: Data Preprocessing & Tokenization

In this notebook, we will load our  dataset and prepare it for the neural network. Since we are doing **Span Identification** (Subtask 1) and **Technique Classification** (Subtask 2), we need to format the text so a Transformer model (like RoBERTa or BERT) can understand it. First let's load the dataset.

In [6]:
drive.mount('/content/drive')

csv_path = "/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/propaganda_train_cleaned.csv"

df = pd.read_csv(csv_path)

df.head()

Mounted at /content/drive


,article_id,technique,start,end,snippet,span_length,full_text_length
0,999000870,Repetition,3812,3831,migrant caravan hea,19,4597
1,111111117,Causal_Oversimplification,671,753,the delay signaled the White House was having ...,82,1064
2,780619695,Repetition,1538,1554,How inconvenient,16,6684
3,780619695,Repetition,1728,1744,How inconvenient,16,6684
4,780619695,Repetition,2018,2034,How inconvenient,16,6684


The next step is to initialize the **Fast Tokenizer**. In the SemEval dataset, the propaganda labels are based on character indices. For example, the dataset tells us that a "Loaded Language" snippet exists from character 45 to character 60 in the raw text. However, RoBERTa doesn't look at characters, it looks at tokens. A standard tokenizer will chop up the text but forget where those tokens originally came from. The Fast Tokenizer returns a special dictionary called **offset_mapping**. This creates a  map linking every single generated token back to its exact character start and end positions in the original text. For this task, we will use RoBERTa.

Generally, a tokenizer first reads the raw string and does a basic split, usually by spaces and punctuation. Then it performs subword splitting, breaksing complex or rare words down into smaller, recognizable chunks. After that the tokenizer looks up every single piece in its massive, pre-trained dictionary and swaps the text chunk for its corresponding ID number. The tokenizer then  injects special structural tokens into the sequence. For example RoBERTa adds `<s>` (Start of Sequence) and `</s>` (End of Sequence). Finally, the tokenizer performs truncation and padding. Neural networks require data to be fed in perfectly uniform batches, like a perfect rectangular matrix. As human sentences are of different lengths, the tokenizer fixes this by cutting off a sentence if it is too long or if it is too short, it fills the rest of the sequence with empty padding tokens (usually ID 1) until it hits the required length.

Let's load the tokenizer and demonstrate what it does.

In [12]:
# 1. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('roberta-base', use_fast=True)

# 2. Process the text
sample_text = "The deeply-corrupt politician lied."
encoded = tokenizer(sample_text)

# 3. Print the results
print(f"Original Text: {sample_text}\n")

# Show the raw mathematical IDs the model actually sees
print(f"Token IDs: {encoded['input_ids']}\n")

# Show the human-readable subword translation
print(f"Tokens: {tokenizer.convert_ids_to_tokens(encoded['input_ids'])}")

Original Text: The deeply-corrupt politician lied.

Token IDs: [0, 133, 4814, 12, 7215, 14709, 8676, 15005, 4, 2]

Tokens: ['<s>', 'The', 'Ġdeeply', '-', 'cor', 'rupt', 'Ġpolitician', 'Ġlied', '.', '</s>']


**The Ġ symbol** represents a space. Ġdeeply has one, but '-' does not have any. That is how the tokenizer knows there was no space before the hyphen.

**cor and rupt** are an example of subword tokenization. The word "corrupt" was not heavily prioritized in the tokenizer's base vocabulary, so it split it into two common subwords. When the model reads this, it pieces the meaning back together.

Now let's demonstrate the "Offset Mapping" on the same sample.

In [13]:
encoded = tokenizer(sample_text, return_offsets_mapping=True)
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
offsets = encoded['offset_mapping']

print("How the Tokenizer maps Tokens back to Characters:")
print(f"{'Token':<15} {'Char Start':<15} {'Char End':<15}")
print("-" * 45)

for token, offset in zip(tokens, offsets):
    print(f"{token:<15} {offset[0]:<15} {offset[1]:<15}")

How the Tokenizer maps Tokens back to Characters:
Token           Char Start      Char End       
---------------------------------------------
<s>             0               0              
The             0               3              
Ġdeeply         4               10             
-               10              11             
cor             11              14             
rupt            14              18             
Ġpolitician     19              29             
Ġlied           30              34             
.               34              35             
</s>            0               0              


Since our CSV only contains the start/end indexes and the labels, we need to load the original article texts to align the tokens. We will group our annotations by article, read the corresponding text file, and run our alignment function.

In [14]:
drive_path = "/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/datasets-v2.tgz"
local_path = "/content/datasets-v2.tgz"

# Retrieve archive from Drive
if os.path.exists(drive_path):
    shutil.copy(drive_path, local_path)
else:
    raise FileNotFoundError(f"Archive not found at {drive_path}. Please check the path.")

# Extract the archive
!tar -xzf {local_path} -C /content/

# Verify extraction
text_files = glob.glob("/content/**/*.txt", recursive=True)
print(f"Extraction complete. Found {len(text_files)} text files in total.")

Extraction complete. Found 1824 text files in total.


Because our neural network predicts a label for every single token, we cannot just label both **cor and rupt** as "Loaded Language". If two separate "Loaded Language" phrases appear right next to each other, the model would not know where one ends and the other begins. We need to introduce the standard formatting used for NLP Token Classification, known as BIO Tagging (Begin, Inside, Outside).


**B-Technique (Begin)**: The very first token of a propaganda phrase.

**I-Technique (Inside)**: Any subsequent tokens that belong to the same phrase.

**O (Outside)**: Normal text that is not propaganda.

No let's tokenize the texts, map your character-level annotations to RoBERTa's token IDs and construct the final dataset for model training.

In [18]:
def align_tokens_and_labels(raw_text, annotations, tokenizer):
    # 1. Create a character-level label array (default to "O")
    char_labels = ["O"] * len(raw_text)

    # 2. Overlay the BIO tags based on annotations
    for ann in annotations:
        start = ann['start']
        end = ann['end']
        technique = ann['technique']

        # Ensure boundaries are within the text
        if start < 0 or end > len(raw_text):
            continue

        # Set the 'B-' tag for the first character
        char_labels[start] = f"B-{technique}"

        # Set the 'I-' tag for the remaining characters in the span
        for i in range(start + 1, end):
            char_labels[i] = f"I-{technique}"

    # 3. Tokenize the text and get character offsets
    # (max_length handles extremely long articles)
    encoded = tokenizer(raw_text, return_offsets_mapping=True, truncation=True, max_length=512)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
    offsets = encoded['offset_mapping']
    input_ids = encoded['input_ids']

    # 4. Align token labels using the first character of each token
    token_labels = []
    for token, offset in zip(tokens, offsets):
        start_char, end_char = offset

        # If it's a special token (like <s> or </s>), offset is (0,0)
        if offset == (0, 0):
            token_labels.append("O")
        else:
            # We map the token to the label of its starting character
            token_labels.append(char_labels[start_char])

    return tokens, token_labels, input_ids

In [19]:
# 1. Find all text files in the extracted dataset
text_files = glob.glob("/content/**/*.txt", recursive=True)
article_files = [f for f in text_files if "article" in os.path.basename(f).lower()]

if not article_files:
    raise FileNotFoundError("Could not find any article text files.")

# 2. Build the ID map
article_file_map = {}
for filepath in article_files:
    filename = os.path.basename(filepath)
    art_id = filename.replace("article", "").replace(".txt", "")
    article_file_map[art_id] = filepath

grouped_annotations = df.groupby('article_id')

all_tokens = []
all_labels = []
all_input_ids = []

print("Processing articles and aligning tokens...")
for article_id, group in tqdm(grouped_annotations):
    str_art_id = str(article_id)

    if str_art_id not in article_file_map:
        continue

    article_path = article_file_map[str_art_id]

    with open(article_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    annotations = group[['start', 'end', 'technique']].to_dict('records')

    # Run the alignment function you defined in the previous cell
    tokens, token_labels, input_ids = align_tokens_and_labels(raw_text, annotations, tokenizer)

    all_tokens.append(tokens)
    all_labels.append(token_labels)
    all_input_ids.append(input_ids)

print(f"Successfully processed {len(all_tokens)} articles.")

Processing articles and aligning tokens...


100%|██████████| 357/357 [00:09<00:00, 38.70it/s]

Successfully processed 357 articles.


Let's map our string tags (like B-Loaded_Language) to unique integer IDs.

In [20]:
# 1. Collect all unique BIO tags generated
unique_labels = set()
for labels in all_labels:
    unique_labels.update(labels)

# 2. Sort labels and keep 'O' at index 0 (standard NLP practice)
unique_labels = sorted(list(unique_labels))
if 'O' in unique_labels:
    unique_labels.remove('O')
    unique_labels = ['O'] + unique_labels

# 3. Build mapping dictionaries
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}

print("Label Mapping is created. Here are the first 5 mappings:")
for i in range(min(5, len(id2label))):
    print(f"{i} : {id2label[i]}")

# 4. Map string labels to integer IDs
all_label_ids = []
for labels in all_labels:
    label_ids = [label2id[label] for label in labels]
    all_label_ids.append(label_ids)

Label Mapping is created. Here are the first 5 mappings:
0 : O
1 : B-Appeal_to_Authority
2 : B-Appeal_to_fear-prejudice
3 : B-Bandwagon,Reductio_ad_hitlerum
4 : B-Black-and-White_Fallacy


Now let's combine the three Python lists (all_input_ids, all_tokens, all_labels) into a standardized Hugging Face Dataset object. This format is optimized for speed and memory, which RoBERTa requires for training.

In [21]:

dataset_dict = {
    "input_ids": all_input_ids,
    "tokens": all_tokens,
    "ner_tags": all_label_ids
}

hf_dataset = Dataset.from_dict(dataset_dict)
print(hf_dataset)

Dataset({
    features: ['input_ids', 'tokens', 'ner_tags'],
    num_rows: 357
})



Now let's save and export for Notebook 03.

In [22]:
# 1. Combine your processed lists into a dictionary
# We name the target column "labels" because RoBERTa automatically looks for that exact name to calculate its loss during training.
data_dict = {
    "input_ids": all_input_ids,
    "tokens": all_tokens,
    "labels": all_label_ids
}

# 2. Convert to a Hugging Face Dataset format
hf_dataset = Dataset.from_dict(data_dict)

print("Hugging Face Dataset successfully created!")
display(hf_dataset)

# 3. Define your Google Drive save path
# (Update this path if your folder is named differently)
save_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda/propaganda_tokenized_dataset'

# 4. Save it permanently to disk
hf_dataset.save_to_disk(save_path)

Hugging Face Dataset successfully created!


Dataset({
    features: ['input_ids', 'tokens', 'labels'],
    num_rows: 357
})

Saving the dataset (0/1 shards):   0%|          | 0/357 [00:00<?, ? examples/s]